## 1. Paths and imports


# TabPFN FP follow-up (VLST): ratio sweep + dual evaluation matrices

1. Loads **full data** (`VLST.csv`) and **union mined FPs** (`false_positives_union_test.csv`), aligns features as before.
2. Reserves a **fixed negative-only test fold** (`TestFixed`) from eligible true negatives (excludes rows referenced by union `test_row_id`). All positives stay available for training.
3. For ratios **r ∈ {1, 5, 10, 15, …}** (until `92·r` negatives exceed the train pool), builds training data: **all positives** in the pool + **mined FPs first** + **other negatives** to reach `n_neg = min(92·r, n_fp + |TN_train_pool|)` without reusing rows. A **final** iteration trains on the **full train pool** (all positives + all union FPs + all remaining eligible negatives) if that is not identical to the last ratio step.
4. Each iteration: **stratified 60/20/20** on the combined matrix → fit **TabPFN** on the train fold, pick threshold **t\*** = argmax **F0.5** on the val fold (same `t_grid` as before). Report **two** confusion matrices at **t\*** only:
   - **split_a** — fixed `TestFixed` (normal test split; negatives only, so positives in truth are absent by construction).
   - **split_b** — `TestFixed` ∪ all **full-table** pool rows not used in the current training draw (positives + negatives + union-row ids accounted for).
5. Writes **`fp_followup_ratio_sweep.csv`** under `OUTPUT_DIR`.

**Env:** `VLST_FULL_DATA_PATH`, `VLST_FULLDATA_TARGET_COL`, `VLST_FP_OUTPUT_DIR` / `VLST_FP_MINING_OUT` / `VLST_FP_RUN_TAG` (union CSV directory), `VLST_FP_FOLLOWUP_OUT_DIR`, `TABPFN_TOKEN`, `VLST_TABPFN_FOLLOWUP_SEED`, `TABPFN_DEVICE`, `TABPFN_N_ESTIMATORS`.


In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = int(os.environ.get("VLST_TABPFN_FOLLOWUP_SEED", "42"))

FULL_DATA_PATH = os.environ.get(
    "VLST_FULL_DATA_PATH",
    "/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv",
)


def pick_output_dir_modeling_fp() -> str:
    out = os.environ.get("VLST_FP_FOLLOWUP_OUT_DIR")
    if out:
        return os.path.expanduser(out.rstrip(os.sep))
    if os.path.isdir("/kaggle/working"):
        return "/kaggle/working/vlst_fp_followup_output"
    return os.path.normpath(os.path.join("..", "..", "data", "result", "modeling_fp"))


OUTPUT_DIR = pick_output_dir_modeling_fp()
os.makedirs(OUTPUT_DIR, exist_ok=True)


def pick_fp_mining_output_dir() -> str:
    out = os.environ.get("VLST_FP_OUTPUT_DIR") or os.environ.get("VLST_FP_MINING_OUT")
    if not out:
        kaggle_fp_out = "/kaggle/input/datasets/amirmahdidaraei/fp-output/vlst_fp_mining_output"
        if os.path.isdir(kaggle_fp_out):
            out = kaggle_fp_out
    if out:
        out = os.path.expanduser(out.rstrip(os.sep))
    elif os.path.isdir("/kaggle/working"):
        out = "/kaggle/working/vlst_fp_mining_output"
    else:
        repo_modeling_fp = os.path.normpath(
            os.path.join("..", "..", "data", "result", "modeling_fp")
        )
        if os.path.isfile(os.path.join(repo_modeling_fp, "false_positives_union_test.csv")):
            out = repo_modeling_fp
        else:
            out = os.path.normpath(
                os.path.join("..", "..", "data", "result", "modeling_advanced", "fp_mining")
            )
    tag = os.environ.get("VLST_FP_RUN_TAG", "").strip()
    if tag:
        out = os.path.join(out, tag)
    return out


FP_MINING_OUT = pick_fp_mining_output_dir()
UNION_CSV = os.path.join(FP_MINING_OUT, "false_positives_union_test.csv")

print("FULL_DATA_PATH:", FULL_DATA_PATH)
print("Follow-up artifacts dir (OUTPUT_DIR):", OUTPUT_DIR)
print("FP mining outputs (union CSV):", FP_MINING_OUT)
print("Union CSV:", UNION_CSV)


In [ ]:
if os.environ.get("TABPFN_TOKEN"):
    print("TABPFN_TOKEN is set.")
else:
    print("Set TABPFN_TOKEN in the environment before running the TabPFN cell below.")


In [3]:
!pip install tabpfn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but 

## 2. Load data, align features, train/test pools, ratio schedule


In [ ]:
if not os.path.isfile(FULL_DATA_PATH):
    raise FileNotFoundError(
        f"Missing full dataset at {FULL_DATA_PATH}. Set VLST_FULL_DATA_PATH before running."
    )
if not os.path.isfile(UNION_CSV):
    raise FileNotFoundError(
        f"Missing {UNION_CSV}. Run fp mining export first or set VLST_FP_OUTPUT_DIR / VLST_FP_RUN_TAG."
    )

df_full = pd.read_csv(FULL_DATA_PATH, low_memory=False)
df_union = pd.read_csv(UNION_CSV, low_memory=False)
if "test_row_id" not in df_union.columns:
    raise ValueError("Union CSV must contain test_row_id.")


def _norm_col(s: str) -> str:
    return "".join(ch.lower() for ch in str(s).strip() if ch.isalnum())


col_map = {_norm_col(c): c for c in df_full.columns}

forced_target = os.environ.get("VLST_FULLDATA_TARGET_COL", "").strip()
if forced_target:
    fk = _norm_col(forced_target)
    if fk not in col_map:
        raise ValueError(
            f"VLST_FULLDATA_TARGET_COL={forced_target!r} not found. "
            f"Available columns include: {list(df_full.columns)[:12]} ..."
        )
    target_col = col_map[fk]
else:
    preferred = [
        "Stent thrombosis",
        "stent_thrombosis",
        "StentThrombosis",
        "target",
        "label",
        "y",
        "class",
        "Outcome",
        "VLST",
    ]
    target_col = None
    for c in preferred:
        k = _norm_col(c)
        if k in col_map:
            cand = col_map[k]
            vals = set(
                pd.to_numeric(df_full[cand], errors="coerce")
                .dropna()
                .astype(int)
                .unique()
                .tolist()
            )
            if vals.issubset({0, 1}) and len(vals) >= 1:
                target_col = cand
                break
    if target_col is None:
        raise ValueError(
            "Could not confidently infer target column. "
            "Set VLST_FULLDATA_TARGET_COL='Stent thrombosis' (or your label column name)."
        )

print("Using target column from full data:", target_col)

y_full = pd.to_numeric(df_full[target_col], errors="coerce").fillna(0).astype(int).to_numpy()
if not set(np.unique(y_full)).issubset({0, 1}):
    raise ValueError("Target column must be binary (0/1).")

X_full_df = df_full.drop(columns=[target_col]).copy()


def _norm_name(s: str) -> str:
    return "".join(ch.lower() for ch in str(s) if ch.isalnum())


_drop_norm = {
    _norm_name("Time since stent implantation"),
    _norm_name("time_since_implantation"),
    _norm_name("time since implantation"),
}
_drop_cols = [c for c in X_full_df.columns if _norm_name(c) in _drop_norm]
if _drop_cols:
    X_full_df = X_full_df.drop(columns=_drop_cols)
    print("Dropped leakage-style feature(s):", _drop_cols)

for c in X_full_df.columns:
    X_full_df[c] = pd.to_numeric(X_full_df[c], errors="coerce")

meta_cols = {
    "test_row_id",
    "y_true",
    "p_winner",
    "p_tabpfn",
    "fp_winner",
    "fp_tabpfn_legacy_t",
    "fp_tabpfn_any_grid_t",
    "source",
    "threshold",
}
shared_features = [c for c in X_full_df.columns if c not in meta_cols]
if not shared_features:
    raise ValueError("No feature columns in full dataset after exclusions.")

_union_feat = [c for c in df_union.columns if c not in meta_cols]
union_only_cols = [c for c in _union_feat if c not in X_full_df.columns]
if union_only_cols:
    print(
        "Note: union CSV has",
        len(union_only_cols),
        "feature column(s) not in full data (omitted from training schema); examples:",
        union_only_cols[:12],
    )

X_full_use = X_full_df[shared_features].copy()
X_union_use = df_union.reindex(columns=shared_features)
for c in shared_features:
    X_union_use[c] = pd.to_numeric(X_union_use[c], errors="coerce")

med = X_full_use.median(numeric_only=True)
X_full_use = X_full_use.fillna(med)
X_union_use = X_union_use.fillna(med)

_union_ids = pd.to_numeric(df_union["test_row_id"], errors="coerce")
_rid = np.where(np.isfinite(_union_ids.to_numpy(dtype=float)), _union_ids.to_numpy(dtype=float), -1.0).astype(np.int64)
_valid_overlay = (_rid >= 0) & (_rid < len(X_full_df))
if int(_valid_overlay.sum()) > 0:
    _vi = np.flatnonzero(_valid_overlay)
    X_union_use.iloc[_vi, :] = X_full_use.iloc[_rid[_vi]].to_numpy(dtype=np.float32, copy=False)
    print(
        "Union FP rows overlaid from full VLST by test_row_id:",
        int(_valid_overlay.sum()),
        "/",
        len(df_union),
    )
else:
    print(
        "Warning: no union test_row_id in [0, n_rows(VLST)); FP rows stay union CSV + median only."
    )

assert (
    X_full_use.shape[1] == len(shared_features) == X_union_use.shape[1]
), (X_full_use.shape[1], len(shared_features), X_union_use.shape[1])

rng = np.random.RandomState(RANDOM_STATE)
pos_idx = np.flatnonzero(y_full == 1)
tn_pool_idx = np.flatnonzero(y_full == 0)
if pos_idx.size == 0:
    raise ValueError("No positives in full dataset.")

union_full_row_ids = set(int(x) for x in df_union["test_row_id"].astype(int).tolist())

# Eligible true negatives for TN fill / test reserve: label 0 and not a union row id
tn_elig = np.array(
    sorted(i for i in tn_pool_idx.tolist() if int(i) not in union_full_row_ids),
    dtype=int,
)
if tn_elig.size < 32:
    tn_elig = np.array(sorted(tn_pool_idx.tolist()), dtype=int)

# Fixed negative-only test split (normal test split)
TEST_NEG_FRAC = float(os.environ.get("VLST_FP_FOLLOWUP_TEST_NEG_FRAC", "0.2"))
if not (0.0 < TEST_NEG_FRAC < 0.5):
    raise ValueError("VLST_FP_FOLLOWUP_TEST_NEG_FRAC must be in (0, 0.5).")
ix_tn = np.arange(len(tn_elig), dtype=int)
tn_train_ix, tn_test_ix = train_test_split(
    ix_tn,
    test_size=TEST_NEG_FRAC,
    random_state=RANDOM_STATE,
    shuffle=True,
)
TestFixed_neg = tn_elig[tn_test_ix]
tn_for_train_pool = tn_elig[tn_train_ix]

# Deterministic shuffle order for consuming extra negatives across ratio steps
perm_tn = rng.permutation(len(tn_for_train_pool))
tn_ordered = tn_for_train_pool[perm_tn]

fp_mat_all = X_union_use.to_numpy(dtype=np.float32)
n_fp = int(fp_mat_all.shape[0])
n_pos = int(pos_idx.size)

X_pos_all = X_full_use.iloc[pos_idx].to_numpy(dtype=np.float32)
y_pos_vec = np.ones(n_pos, dtype=int)

max_neg_total = n_fp + int(tn_ordered.size)
max_r = max_neg_total // n_pos if n_pos else 0
ratio_candidates = [1] + list(range(5, max(max_r, 5) + 1, 5))
RATIOS = []
for r in ratio_candidates:
    need = n_pos * r
    if need <= max_neg_total:
        RATIOS.append(int(r))

if not RATIOS:
    raise RuntimeError("No valid ratio steps (check TN pool and n_fp).")

# Full-pool negative count (all FPs + all train-pool TNs)
n_neg_full = n_fp + int(tn_ordered.size)

split_summary = {
    "n_pos": n_pos,
    "n_fp_union": n_fp,
    "n_tn_train_pool": int(tn_ordered.size),
    "n_TestFixed_neg": int(TestFixed_neg.size),
    "max_neg_total": max_neg_total,
    "ratios": RATIOS,
    "n_neg_full_pool": int(n_neg_full),
    "TEST_NEG_FRAC": TEST_NEG_FRAC,
}
print("Split / ratio summary:", split_summary)
print("Features (shared_features):", len(shared_features))


def stratified_602020(X_mat: np.ndarray, y_vec: np.ndarray, random_state: int):
    idx = np.arange(len(y_vec), dtype=int)
    i_tr, i_temp = train_test_split(
        idx,
        test_size=0.4,
        random_state=random_state,
        shuffle=True,
        stratify=y_vec,
    )
    i_va, i_ho = train_test_split(
        i_temp,
        test_size=0.5,
        random_state=random_state,
        shuffle=True,
        stratify=y_vec[i_temp],
    )
    return (
        X_mat[i_tr],
        y_vec[i_tr],
        X_mat[i_va],
        y_vec[i_va],
        X_mat[i_ho],
        y_vec[i_ho],
    )


def build_train_arrays(n_neg_target: int, rng_step: np.random.RandomState):
    # All positives + FPs first + TN fill; returns arrays and bookkeeping.
    need = int(min(n_neg_target, n_fp + tn_ordered.size))
    n_fp_take = min(n_fp, need)
    need_after_fp = need - n_fp_take
    X_fp_part = fp_mat_all[:n_fp_take] if n_fp_take > 0 else np.empty((0, fp_mat_all.shape[1]), dtype=np.float32)
    tn_take = int(min(need_after_fp, tn_ordered.size))
    if tn_take > 0:
        tn_idx_used = tn_ordered[:tn_take]
        X_tn_part = X_full_use.iloc[tn_idx_used].to_numpy(dtype=np.float32)
    else:
        tn_idx_used = np.array([], dtype=int)
        X_tn_part = np.empty((0, fp_mat_all.shape[1]), dtype=np.float32)

    X_all = np.vstack([X_pos_all, X_fp_part, X_tn_part]).astype(np.float32)
    y_all = np.concatenate(
        [
            y_pos_vec,
            np.zeros(n_fp_take, dtype=int),
            np.zeros(tn_take, dtype=int),
        ]
    )
    perm = rng_step.permutation(len(y_all))
    X_all, y_all = X_all[perm], y_all[perm]

    # Row-id sets for matrix B (full-table indices)
    fp_row_ids_used = set()
    for j in range(n_fp_take):
        rid = int(df_union.iloc[j]["test_row_id"])
        fp_row_ids_used.add(rid)
    train_full_idx = set(int(x) for x in pos_idx.tolist()) | set(int(x) for x in tn_idx_used.tolist()) | fp_row_ids_used
    return X_all, y_all, n_fp_take, tn_idx_used, train_full_idx


def pool_full_idx_set():
    return (
        set(int(x) for x in pos_idx.tolist())
        | set(int(x) for x in tn_elig.tolist())
        | set(int(x) for x in union_full_row_ids)
    )


def Xy_for_full_indices(indices: np.ndarray):
    # Features/labels for full-table row indices.
    indices = np.asarray(indices, dtype=int)
    Xb = X_full_use.iloc[indices].to_numpy(dtype=np.float32)
    yb = y_full[indices].astype(int)
    return Xb, yb


def matrix_b_indices(train_full_idx: set):
    pool = pool_full_idx_set()
    return np.asarray(sorted(pool - set(train_full_idx)), dtype=int)



## 3. Ratio sweep — TabPFN, F0.5 threshold, dual confusion matrices + CSV


In [ ]:
token = os.environ.get("TABPFN_TOKEN")
if not token:
    raise RuntimeError(
        "Set TABPFN_TOKEN (shell export, Kaggle secret, or os.environ) before running TabPFN."
    )

try:
    import torch
except ImportError:
    torch = None

TABPFN_N_ESTIMATORS = int(os.environ.get("TABPFN_N_ESTIMATORS", "8"))
TABPFN_DEVICE = os.environ.get("TABPFN_DEVICE")
if not TABPFN_DEVICE:
    TABPFN_DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"

from tabpfn import TabPFNClassifier

t_grid = np.arange(0.01, 1.0, 0.01)


def best_threshold(y_true, p, grid, metric_fn):
    best_t, best_v = 0.5, -1.0
    for t in grid:
        y_hat = (p >= t).astype(int)
        v = float(metric_fn(y_true, y_hat))
        if v > best_v:
            best_v, best_t = v, float(t)
    return best_t, best_v


def f05m(y, yhat):
    return fbeta_score(y, yhat, beta=0.5, zero_division=0)


def metric_bundle(y_true, y_hat, p):
    out = {
        "precision": float(precision_score(y_true, y_hat, zero_division=0)),
        "recall": float(recall_score(y_true, y_hat, zero_division=0)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "f05": float(fbeta_score(y_true, y_hat, beta=0.5, zero_division=0)),
        "f2": float(fbeta_score(y_true, y_hat, beta=2.0, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_hat)),
    }
    try:
        out["roc_auc"] = float(roc_auc_score(y_true, p))
    except ValueError:
        out["roc_auc"] = float("nan")
    try:
        out["pr_auc"] = float(average_precision_score(y_true, p))
    except ValueError:
        out["pr_auc"] = float("nan")
    return out


def print_confusion_matrix_text(cm):
    cm = np.asarray(cm, dtype=int)
    print("Confusion matrix (rows=true, cols=pred):")
    if cm.shape == (2, 2):
        print("                pred:0 (neg)   pred:1 (pos)")
        print(f"    true:0 (neg)   {cm[0, 0]:>10}   {cm[0, 1]:>10}   TN, FP")
        print(f"    true:1 (pos)   {cm[1, 0]:>10}   {cm[1, 1]:>10}   FN, TP")
    else:
        for row in cm:
            print("   ", " ".join(f"{v:>8}" for v in row))


def report_cm(name, y_true, y_hat, p):
    print("\n===", name, "@ t*(val F0.5) ===")
    mb = metric_bundle(y_true, y_hat, p)
    print("ROC-AUC:", round(mb["roc_auc"], 4), "| PR-AUC:", round(mb["pr_auc"], 4))
    print("Accuracy:", round(mb["accuracy"], 4))
    print(
        "Precision / Recall / F1 / F0.5:",
        round(mb["precision"], 4),
        round(mb["recall"], 4),
        round(mb["f1"], 4),
        round(mb["f05"], 4),
    )
    print(classification_report(y_true, y_hat, digits=4, zero_division=0))
    print_confusion_matrix_text(confusion_matrix(y_true, y_hat, labels=[0, 1]))
    return mb


sweep_rows = []
step_i = 0

# --- Ratio iterations ---
for r in RATIOS:
    n_neg_target = n_pos * int(r)
    rs = np.random.RandomState(RANDOM_STATE + step_i * 9973)
    X_all, y_all, n_fp_take, tn_used, train_full_idx = build_train_arrays(n_neg_target, rs)
    X_tr, y_tr, X_va, y_va, X_ho, y_ho = stratified_602020(X_all, y_all, RANDOM_STATE + step_i)

    clf = TabPFNClassifier(
        random_state=RANDOM_STATE,
        n_estimators=TABPFN_N_ESTIMATORS,
        device=TABPFN_DEVICE,
        ignore_pretraining_limits=True,
        balance_probabilities=True,
    )
    clf.fit(X_tr, y_tr)
    p_va = clf.predict_proba(X_va)[:, 1]
    t_star, v_f05_va = best_threshold(y_va, p_va, t_grid, f05m)

    X_a, y_a = Xy_for_full_indices(TestFixed_neg)
    p_a = clf.predict_proba(X_a)[:, 1]
    y_hat_a = (p_a >= t_star).astype(int)
    mb_a = report_cm(f"ratio={r} | split_a (TestFixed_neg)", y_a, y_hat_a, p_a)

    b_ix = matrix_b_indices(train_full_idx)
    X_b, y_b = Xy_for_full_indices(b_ix)
    p_b = clf.predict_proba(X_b)[:, 1]
    y_hat_b = (p_b >= t_star).astype(int)
    mb_b = report_cm(f"ratio={r} | split_b (TestFixed + left-out pool)", y_b, y_hat_b, p_b)

    cm_a = confusion_matrix(y_a, y_hat_a, labels=[0, 1])
    cm_b = confusion_matrix(y_b, y_hat_b, labels=[0, 1])
    row = {
        "step_kind": "ratio",
        "ratio_requested": int(r),
        "n_neg_target": int(n_neg_target),
        "n_fp_in_constructed": int(n_fp_take),
        "n_tn_in_constructed": int(len(tn_used)),
        "n_fit_train": int(X_tr.shape[0]),
        "n_fit_val": int(X_va.shape[0]),
        "t_star_val_f05": float(t_star),
        "val_f05_at_t_star": float(v_f05_va),
        "split_a_n": int(len(y_a)),
        "split_b_n": int(len(y_b)),
        "cm_a_00": int(cm_a[0, 0]),
        "cm_a_01": int(cm_a[0, 1]),
        "cm_a_10": int(cm_a[1, 0]) if cm_a.shape[0] > 1 else 0,
        "cm_a_11": int(cm_a[1, 1]) if cm_a.shape[0] > 1 else 0,
        "cm_b_00": int(cm_b[0, 0]),
        "cm_b_01": int(cm_b[0, 1]),
        "cm_b_10": int(cm_b[1, 0]) if cm_b.shape[0] > 1 else 0,
        "cm_b_11": int(cm_b[1, 1]) if cm_b.shape[0] > 1 else 0,
        **{f"a_{k}": float(v) for k, v in mb_a.items()},
        **{f"b_{k}": float(v) for k, v in mb_b.items()},
    }
    sweep_rows.append(row)
    step_i += 1

# --- Full pool iteration (if not already identical to last ratio) ---
rs = np.random.RandomState(RANDOM_STATE + step_i * 9973)
X_all_f, y_all_f, n_fp_f, tn_used_f, train_full_idx_f = build_train_arrays(n_neg_full, rs)
last_same = False
if sweep_rows:
    last = sweep_rows[-1]
    last_same = (
        int(last["n_fp_in_constructed"]) == int(n_fp_f)
        and int(last["n_tn_in_constructed"]) == int(len(tn_used_f))
    )
if not last_same:
    X_tr, y_tr, X_va, y_va, _, _ = stratified_602020(X_all_f, y_all_f, RANDOM_STATE + step_i)
    clf = TabPFNClassifier(
        random_state=RANDOM_STATE,
        n_estimators=TABPFN_N_ESTIMATORS,
        device=TABPFN_DEVICE,
        ignore_pretraining_limits=True,
        balance_probabilities=True,
    )
    clf.fit(X_tr, y_tr)
    p_va = clf.predict_proba(X_va)[:, 1]
    t_star, v_f05_va = best_threshold(y_va, p_va, t_grid, f05m)

    X_a, y_a = Xy_for_full_indices(TestFixed_neg)
    p_a = clf.predict_proba(X_a)[:, 1]
    y_hat_a = (p_a >= t_star).astype(int)
    mb_a = report_cm("full_pool | split_a (TestFixed_neg)", y_a, y_hat_a, p_a)

    b_ix = matrix_b_indices(train_full_idx_f)
    X_b, y_b = Xy_for_full_indices(b_ix)
    p_b = clf.predict_proba(X_b)[:, 1]
    y_hat_b = (p_b >= t_star).astype(int)
    mb_b = report_cm("full_pool | split_b (TestFixed + left-out pool)", y_b, y_hat_b, p_b)

    cm_a = confusion_matrix(y_a, y_hat_a, labels=[0, 1])
    cm_b = confusion_matrix(y_b, y_hat_b, labels=[0, 1])
    sweep_rows.append(
        {
            "step_kind": "full_pool",
            "ratio_requested": -1,
            "n_neg_target": int(n_neg_full),
            "n_fp_in_constructed": int(n_fp_f),
            "n_tn_in_constructed": int(len(tn_used_f)),
            "n_fit_train": int(X_tr.shape[0]),
            "n_fit_val": int(X_va.shape[0]),
            "t_star_val_f05": float(t_star),
            "val_f05_at_t_star": float(v_f05_va),
            "split_a_n": int(len(y_a)),
            "split_b_n": int(len(y_b)),
            "cm_a_00": int(cm_a[0, 0]),
            "cm_a_01": int(cm_a[0, 1]),
            "cm_a_10": int(cm_a[1, 0]) if cm_a.shape[0] > 1 else 0,
            "cm_a_11": int(cm_a[1, 1]) if cm_a.shape[0] > 1 else 0,
            "cm_b_00": int(cm_b[0, 0]),
            "cm_b_01": int(cm_b[0, 1]),
            "cm_b_10": int(cm_b[1, 0]) if cm_b.shape[0] > 1 else 0,
            "cm_b_11": int(cm_b[1, 1]) if cm_b.shape[0] > 1 else 0,
            **{f"a_{k}": float(v) for k, v in mb_a.items()},
            **{f"b_{k}": float(v) for k, v in mb_b.items()},
        }
    )
else:
    print("Full-pool step skipped (identical to last ratio iteration).")

sweep_df = pd.DataFrame(sweep_rows)
_sweep_path = os.path.join(OUTPUT_DIR, "fp_followup_ratio_sweep.csv")
sweep_df.to_csv(_sweep_path, index=False)
print("\nTabPFN device:", TABPFN_DEVICE)
print("Saved:", _sweep_path, "rows:", len(sweep_df))


## 4. Output


In [ ]:
_p = os.path.join(OUTPUT_DIR, "fp_followup_ratio_sweep.csv")
print("Primary sweep table:", _p)
if len(sweep_df):
    print(sweep_df.to_string())


## 5. Saved artifact paths


In [ ]:
print(
    "Artifacts written to:",
    OUTPUT_DIR,
    "\n",
    " - fp_followup_ratio_sweep.csv",
)
